# S09. 몬테카를로 실험 — 난수로 원주율과 측정 불확도 구하기

| 항목 | 내용 |
|---|---|
| 권장 시기 | 9주차 (17회 이후) |
| 시간 | 60분 |
| 변환한 R 코드 | `4. R/3. 코드 연습/1. 일자별 연습/Test/실험_몬테카를로.R` |
| 필요 라이브러리 | `matplotlib` (11회에 설치) |

**몬테카를로 방법:** 정답을 공식으로 구하기 어려울 때, **무작위 실험을 아주 많이 반복**해서 답을 추정하는 방법입니다.
측정 불확도 평가(GUM 부록 Supplement 1)에서도 쓰는 방법입니다.

> ✅ **해답 노트북** — 연습 문제의 예시 답안이 채워져 있습니다.

## 1. 원주율 추정 — R 코드 옮기기
```r
tibble(x = runif(n=10000, min=-1, max=1),
       y = runif(n=10000, min=-1, max=1),
       in_circle = ifelse(x^2 + y^2 <= 1, 1, 0))
```
- 한 변이 2인 정사각형 안에 점을 무작위로 찍으면, 반지름 1인 원 안에 들어갈 확률 = 원 넓이 ÷ 정사각형 넓이 = π ÷ 4
- 그래서 **π ≈ 4 × (원 안의 점 수 ÷ 전체 점 수)**

In [ ]:
import random

random.seed(2023)            # 같은 결과가 나오도록 고정 (실험 재현)
n = 10000
xs = [random.uniform(-1, 1) for _ in range(n)]       # R의 runif(n, -1, 1)
ys = [random.uniform(-1, 1) for _ in range(n)]
inside = [x * x + y * y <= 1 for x, y in zip(xs, ys)]

pi_est = 4 * sum(inside) / n
print("추정 π:", pi_est)

In [ ]:
import matplotlib.pyplot as plt

colors = ["#79c3d6" if i else "#cdcdcd" for i in inside]
plt.figure(figsize=(4.5, 4.5))
plt.scatter(xs, ys, c=colors, s=1)
plt.gca().set_aspect("equal")                # R의 coord_equal()
plt.title(f"Monte Carlo pi = {pi_est:.4f}")
plt.show()

### 🧪 연습 1 — 반복 횟수와 정확도
`n`을 100, 1,000, 10,000, 100,000으로 바꿔 π를 추정하고, **실제 π(3.14159…)와의 차이**를 출력하세요. `n`이 커지면 어떻게 되나요?
(힌트: 17회처럼 `estimate_pi(n)` 함수를 만들고 반복문으로 부르기, `import math` → `math.pi`)

In [ ]:
import math

def estimate_pi(n):
    hits = 0
    for _ in range(n):
        x, y = random.uniform(-1, 1), random.uniform(-1, 1)
        if x * x + y * y <= 1:
            hits += 1
    return 4 * hits / n

random.seed(1)
for n in [100, 1000, 10000, 100000]:
    est = estimate_pi(n)
    print(f"n={n:>7}: {est:.4f}  (차이 {abs(est - math.pi):.4f})")

## 2. 업무 적용 — 측정 결과의 불확도 실험
시료 농도를 이렇게 계산한다고 합시다. (수업용 단순 모델)
```
농도 = 기기 측정값 × 최종 부피 ÷ 시료 부피
```
각 값에는 **오차**가 있습니다. (정규분포, 표준편차로 표현)

| 값 | 참값(가정) | 표준 불확도 |
|---|---|---|
| 기기 측정값 | 2.50 mg/L | 2 % |
| 최종 부피 | 50.0 mL | 0.5 % |
| 시료 부피 | 10.0 mL | 1 % |

각 값을 오차만큼 흔들어 **10,000번 계산**하면, 최종 농도의 **퍼짐(불확도)** 을 볼 수 있습니다.

In [ ]:
import statistics as st

random.seed(42)
N = 10000
results = []
for _ in range(N):
    reading = random.gauss(2.50, 2.50 * 0.02)     # 평균, 표준편차
    v_final = random.gauss(50.0, 50.0 * 0.005)
    v_sample = random.gauss(10.0, 10.0 * 0.01)
    results.append(reading * v_final / v_sample)

mean = st.mean(results)
sd = st.stdev(results)
results.sort()
low, high = results[int(N * 0.025)], results[int(N * 0.975)]
print(f"평균 {mean:.3f} mg/L, 표준편차 {sd:.3f} (상대 {sd / mean * 100:.2f} %)")
print(f"95 % 범위: {low:.3f} ~ {high:.3f} mg/L")

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.hist(results, bins=50, color="#4472C4")
plt.axvline(low, color="red", linestyle="--")
plt.axvline(high, color="red", linestyle="--")
plt.xlabel("Concentration (mg/L)")
plt.ylabel("Count")
plt.title("Monte Carlo uncertainty (10,000 runs)")
plt.tight_layout()
plt.show()

### 공식으로 구한 값과 비교
곱하기·나누기만 있는 식에서는 **상대 불확도의 제곱합의 제곱근**으로 합성 불확도를 근사할 수 있습니다.
```
√(2² + 0.5² + 1²) ≈ 2.29 %
```
몬테카를로 결과(상대 표준편차)와 비슷한지 확인하세요.

### 🧪 연습 2
시료 부피의 불확도를 **1 % → 3 %** 로 키우면 최종 상대 불확도는 얼마나 커지나요? 몬테카를로로 구하고 공식 값 `√(2² + 0.5² + 3²)`과 비교하세요.

In [ ]:
def mc_uncertainty(u_reading, u_vfinal, u_vsample, n=10000, seed=7):
    random.seed(seed)
    out = []
    for _ in range(n):
        r = random.gauss(2.50, 2.50 * u_reading / 100)
        vf = random.gauss(50.0, 50.0 * u_vfinal / 100)
        vs = random.gauss(10.0, 10.0 * u_vsample / 100)
        out.append(r * vf / vs)
    return st.stdev(out) / st.mean(out) * 100

print(f"몬테카를로: {mc_uncertainty(2, 0.5, 3):.2f} %")
print(f"공식 근사 : {math.sqrt(2**2 + 0.5**2 + 3**2):.2f} %")

---
## 정리
- 몬테카를로 = **무작위 실험을 많이 반복**해서 답을 추정
- `random.seed(...)` 로 **재현 가능**하게 (보고서에 쓸 결과라면 필수)
- 반복이 많을수록 정확해지지만 시간이 걸림
- 측정 불확도 평가에서 공식이 복잡할 때 유용 — 실제 업무 적용 시에는 **기관의 불확도 산정 절차**를 따르세요